# 🎤 WABƲƐ — Dictée ikposso (« Parle, WABƲƐ écrit »)

Ce carnet apprend à un modèle de reconnaissance vocale à **comprendre l'ikposso**.

- Données : corpus **WAXAL** (Google Research et partenaires, dont l'Université du Ghana), partie ikposso `kpo`, licence **CC-BY-4.0**.
- Modèle de départ : **MMS** de Meta (`facebook/mms-1b-all`), on entraîne seulement un petit « adaptateur » ikposso.
- Orthographe : celle de WABƲƐ (ʋ, ɩ).

## Mode d'emploi
1. Menu **Exécution → Modifier le type d'exécution → GPU (T4)** → Enregistrer.
2. Menu **Exécution → Tout exécuter**.
3. Laisser l'onglet ouvert (2 à 4 heures). À la fin, le modèle est enregistré dans votre **Google Drive** (dossier `WABUE-dictee-ikposso`) et une démo avec micro s'affiche en bas.

Source des données à citer : *WAXAL, Google Research (2025), CC-BY-4.0.*


In [ ]:
# 1. Installer les outils
!pip install -q jiwer soundfile librosa gradio  # transformers, datasets, accelerate : versions déjà installées dans Colab


In [ ]:
# 2. Vérifier qu'on a bien un GPU
import torch
assert torch.cuda.is_available(), "Pas de GPU : menu Exécution > Modifier le type d'exécution > GPU (T4)"
print("GPU :", torch.cuda.get_device_name(0))


In [ ]:
# 3. Réglages (on peut augmenter N_TRAIN si le GPU est plus puissant)
N_TRAIN_FILES = 8      # fichiers d'entraînement WAXAL (env. 600 phrases chacun, 24 au total)
N_TEST_FILES  = 1      # fichiers de test (env. 460 phrases chacun)
N_TEST_MAX    = 300    # phrases de test utilisées pendant l'entraînement
EPOCHS        = 3
MAX_SECONDES  = 30     # on ignore les audios plus longs (WAXAL : phrases parlées assez longues)
SORTIE        = "/content/wabue-dictee-ikposso"


In [ ]:
# 4. Télécharger la partie ikposso de WAXAL
# Si Hugging Face demande une connexion, décommentez les 2 lignes suivantes et collez votre jeton (gratuit) :
# from huggingface_hub import notebook_login
# notebook_login()
from huggingface_hub import hf_hub_download
from datasets import load_dataset

def fichiers(partie, n, total):
    return [hf_hub_download("google/WaxalNLP", f"data/ASR/kpo/{partie}-{i:05d}-of-{total:05d}.parquet", repo_type="dataset") for i in range(n)]

train_files = fichiers("train", N_TRAIN_FILES, 24)
test_files  = fichiers("test",  N_TEST_FILES, 4)
ds = load_dataset("parquet", data_files={"train": train_files, "test": test_files})
print(ds)


In [ ]:
# 5. Trouver la colonne audio et la mettre à 16 kHz
from datasets import Audio
cols = ds["train"].column_names
audio_col = next((c for c,f in ds["train"].features.items() if isinstance(f, Audio)), None)
if audio_col is None:
    audio_col = next(c for c in cols if "audio" in c.lower())
print("Colonne audio :", audio_col)
ds = ds.cast_column(audio_col, Audio(sampling_rate=16000))
if audio_col != "audio":
    ds = ds.rename_column(audio_col, "audio")
ds["test"] = ds["test"].select(range(min(N_TEST_MAX, len(ds["test"]))))


In [ ]:
# 6. Mettre le texte dans l'orthographe WABƲƐ (ʋ, ɩ) et nettoyer la ponctuation
import re, unicodedata
REMPLACE = {"ʊ": "ʋ", "Ʊ": "Ʋ", "ɪ": "ɩ", "ι": "ɩ", "Ɪ": "Ɩ"}
PONCT = re.compile(r"[\.,;:!\?\"“”«»\(\)\[\]…–—\-_/']")

def normalise(t):
    t = unicodedata.normalize("NFD", t or "")
    for a, b in REMPLACE.items():
        t = t.replace(a, b)
    t = unicodedata.normalize("NFC", t).lower()
    t = PONCT.sub(" ", t)
    return re.sub(r"\s+", " ", t).strip()

ds = ds.map(lambda b: {"texte": normalise(b["transcription"])})
ds = ds.filter(lambda b: len(b["texte"]) > 0)
for i in range(5):
    print(ds["train"][i]["texte"])


In [ ]:
# 7. Construire l'alphabet ikposso du modèle
import json, os
lettres = set()
for t in list(ds["train"]["texte"]) + list(ds["test"]["texte"]):
    lettres.update(t)
vocab = {c: i for i, c in enumerate(sorted(lettres))}
vocab["|"] = vocab.pop(" ")
vocab["[UNK]"] = len(vocab)
vocab["[PAD]"] = len(vocab)
os.makedirs(SORTIE, exist_ok=True)
with open(f"{SORTIE}/vocab.json", "w", encoding="utf-8") as f:
    json.dump({"kpo": vocab}, f, ensure_ascii=False)
print(len(vocab), "symboles :", " ".join(sorted(vocab)))


In [ ]:
# 8. Préparer le processeur (texte + son)
import numpy as np
from transformers import Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor
tokenizer = Wav2Vec2CTCTokenizer(f"{SORTIE}/vocab.json", unk_token="[UNK]", pad_token="[PAD]", word_delimiter_token="|", target_lang="kpo")
feature_extractor = Wav2Vec2FeatureExtractor(feature_size=1, sampling_rate=16000, padding_value=0.0, do_normalize=True, return_attention_mask=True)
processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)

def prepare(b):
    a = b["audio"]
    b["input_values"] = processor(a["array"], sampling_rate=16000).input_values[0]
    b["duree"] = len(a["array"]) / 16000
    b["labels"] = processor(text=b["texte"]).input_ids
    return b

ds = ds.map(prepare, remove_columns=ds["train"].column_names)
d = np.array(list(ds["train"]["duree"]))
print(f"Durées : moyenne {d.mean():.1f} s, médiane {np.median(d):.1f} s, max {d.max():.1f} s")
print(f"Phrases gardées (< {MAX_SECONDES} s) : {(d < MAX_SECONDES).sum()} sur {len(d)}")
ds = ds.filter(lambda x: x < MAX_SECONDES, input_columns=["duree"])
assert len(ds["train"]) > 500, "Trop peu de phrases gardées : augmentez MAX_SECONDES au bloc 3"
print(ds)


In [ ]:
# 9. Assembler les lots (padding)
from dataclasses import dataclass
from typing import Dict, List, Union

@dataclass
class Collator:
    processor: Wav2Vec2Processor
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        inp = [{"input_values": f["input_values"]} for f in features]
        lab = [{"input_ids": f["labels"]} for f in features]
        batch = self.processor.pad(inp, padding=True, return_tensors="pt")
        labels_batch = self.processor.pad(labels=lab, padding=True, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

collator = Collator(processor=processor)


In [ ]:
# 10. Mesure de qualité : taux d'erreur sur les mots (WER) et les lettres (CER)
import numpy as np, jiwer

def metriques(pred):
    ids = np.argmax(pred.predictions, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id
    p = processor.batch_decode(ids)
    r = processor.batch_decode(pred.label_ids, group_tokens=False)
    paires = [(a, b) for a, b in zip(r, p) if a.strip()]
    r2 = [a for a, _ in paires]; p2 = [b if b.strip() else "-" for _, b in paires]
    return {"wer": jiwer.wer(r2, p2), "cer": jiwer.cer(r2, p2)}


In [ ]:
# 11. Charger MMS et ajouter un adaptateur ikposso (seul l'adaptateur apprend)
from transformers import Wav2Vec2ForCTC
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/mms-1b-all",
    attention_dropout=0.0, hidden_dropout=0.0, feat_proj_dropout=0.0, layerdrop=0.0,
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
    ignore_mismatched_sizes=True,
)
model.init_adapter_layers()
model.freeze_base_model()
for p in model._get_adapters().values():
    p.requires_grad = True
print("Paramètres entraînés :", sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
# 12. Entraîner (2 à 4 heures sur un GPU T4 gratuit)
import inspect
from transformers import TrainingArguments, Trainer
_ta = inspect.signature(TrainingArguments.__init__).parameters
_eval_key = "eval_strategy" if "eval_strategy" in _ta else "evaluation_strategy"
reglages = dict(
    output_dir=SORTIE,
    group_by_length=True,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    gradient_checkpointing=True,
    fp16=True,
    save_steps=200, eval_steps=200, logging_steps=50,
    learning_rate=1e-3, warmup_steps=100,
    save_total_limit=2,
    report_to="none",
)
reglages[_eval_key] = "steps"
# on garde seulement les réglages connus par la version de Colab
args = TrainingArguments(**{k: v for k, v in reglages.items() if k in _ta})
_tr = inspect.signature(Trainer.__init__).parameters
_proc = {"processing_class": processor.feature_extractor} if "processing_class" in _tr else {"tokenizer": processor.feature_extractor}
trainer = Trainer(model=model, data_collator=collator, args=args, compute_metrics=metriques,
                  train_dataset=ds["train"], eval_dataset=ds["test"], **_proc)
trainer.train()
print(trainer.evaluate())


In [ ]:
# 13. Enregistrer le modèle dans Google Drive
from safetensors.torch import save_file
save_file({k: v.detach().cpu().contiguous() for k, v in model._get_adapters().items()}, os.path.join(SORTIE, "adapter.kpo.safetensors"), metadata={"format": "pt"})
model.save_pretrained(SORTIE)
processor.save_pretrained(SORTIE)

from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/WABUE-dictee-ikposso"
!cp -r {SORTIE}/*.json {SORTIE}/*.safetensors "/content/drive/MyDrive/WABUE-dictee-ikposso/" 2>/dev/null
print("✅ Modèle enregistré dans Google Drive > WABUE-dictee-ikposso")


In [ ]:
# 14. Exemples : ce que le modèle entend vs le texte réel
model.eval()
for i in range(5):
    ex = ds["test"][i]
    with torch.no_grad():
        logits = model(torch.tensor([ex["input_values"]]).to(model.device)).logits
    pred = processor.decode(torch.argmax(logits, dim=-1)[0])
    vrai = processor.decode(ex["labels"], group_tokens=False)
    print("Réel  :", vrai); print("Modèle:", pred); print("---")


In [ ]:
# 15. Démo : parlez ikposso dans le micro, le modèle écrit
import gradio as gr, librosa
def dicter(chemin):
    if chemin is None:
        return ""
    son, _ = librosa.load(chemin, sr=16000)
    entree = processor(son, sampling_rate=16000, return_tensors="pt").input_values.to(model.device)
    with torch.no_grad():
        logits = model(entree).logits
    return processor.decode(torch.argmax(logits, dim=-1)[0])

gr.Interface(fn=dicter, inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
             outputs="text", title="WABƲƐ — Dictée ikposso (essai)").launch(share=True)
